<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook streams **~300K real Amazon reviews** through 10 cloud workers on Burla, scores them for profanity + screaming + rage-vs-stars mismatch, and surfaces the most unhinged. In about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

The full demo streams **571 million Amazon reviews** (275 GB of JSONL on HuggingFace) through 500+ Burla workers in 3 minutes and builds the *Wall of Fucked Up* — a ranked shrine to the most unhinged reviews ever written. See the [live site](https://burla-cloud.github.io/amazon-review-distiller/).

In this notebook we run the exact same scoring pipeline on **a single category** (`Video_Games`, ~300K reviews) across **10 workers**. Each worker grabs a byte-range chunk of the JSONL, streams rows, scores each review (profanity, caps ratio, exclamation runs, 5-star-with-rage), and returns the top-K per bucket. The client merges the per-worker top-K's into a global top-10.

No LLM. Every score is regex + word counts. Every string is a real, verbatim review.

## 1)&nbsp; Boot some machines (10+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Stream & score 300K Amazon reviews 🚀

We partition the `Video_Games.jsonl` file on HuggingFace into 10 byte-range chunks. Each worker opens an HTTP Range request against the HF CDN, streams the JSONL rows in its range, scores them locally, and returns a compact top-K list. Nothing is downloaded to the notebook's disk.

In [ ]:
import requests  # noqa: F401  # top-level import -> Burla installs requests on workers
from burla import remote_parallel_map

HF_URL = (
    "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/"
    "resolve/main/raw/review_categories/Video_Games.jsonl"
)

# Partition the file into 10 byte-range chunks
head = requests.head(HF_URL, allow_redirects=True, timeout=30)
total_bytes = int(head.headers["content-length"])
# Use only the first ~500 MB for the demo (the full file is ~4 GB)
demo_bytes = min(total_bytes, 500 * 1024 * 1024)
N_CHUNKS = 10
span = demo_bytes // N_CHUNKS
jobs = [(HF_URL, i * span, (i + 1) * span if i < N_CHUNKS - 1 else demo_bytes, i) for i in range(N_CHUNKS)]
print(f"streaming ~{demo_bytes / 1e6:.0f} MB across {N_CHUNKS} byte-range chunks")


def score_chunk(job: tuple) -> dict:
    import json
    import re
    import requests

    url, start, end, chunk_id = job

    PROFANITY = set("fuck fucking shit crap damn hell bullshit asshole bastard piss".split())
    PROFANITY_RE = re.compile(r"\b(" + "|".join(PROFANITY) + r")\b", re.I)
    EXCLAIM_RUN_RE = re.compile(r"!{3,}")

    headers = {"Range": f"bytes={start}-{end - 1}"}
    r = requests.get(url, headers=headers, stream=True, timeout=60)
    r.raise_for_status()

    top_profane = []  # (score, review)
    top_screaming = []
    top_rage_5star = []
    n_rows = 0
    buf = b""
    for chunk in r.iter_content(chunk_size=65536):
        buf += chunk
        lines = buf.split(b"\n")
        buf = lines.pop()  # keep partial last line
        for raw in lines:
            if not raw.strip():
                continue
            try:
                row = json.loads(raw)
            except Exception:
                continue
            n_rows += 1
            text = (row.get("text") or "")[:4000]
            if not text:
                continue
            profanity_hits = len(PROFANITY_RE.findall(text))
            caps_ratio = sum(c.isupper() for c in text) / max(1, sum(c.isalpha() for c in text))
            exclaim_runs = len(EXCLAIM_RUN_RE.findall(text))
            rating = row.get("rating", 0)

            compact = {"rating": rating, "text": text, "asin": row.get("asin")}
            if profanity_hits:
                top_profane.append((profanity_hits, compact))
            if caps_ratio > 0.6 and len(text) > 60:
                top_screaming.append((len(text) * caps_ratio, compact))
            if rating in (5, 5.0) and (profanity_hits + exclaim_runs) > 0:
                top_rage_5star.append((profanity_hits * 3 + exclaim_runs, compact))

    def topk(xs, k=5):
        return sorted(xs, key=lambda x: x[0], reverse=True)[:k]

    return {
        "chunk_id": chunk_id,
        "rows": n_rows,
        "top_profane": topk(top_profane),
        "top_screaming": topk(top_screaming),
        "top_rage_5star": topk(top_rage_5star),
    }


# 10 chunks -> 10 workers streaming + scoring in parallel
chunk_results = remote_parallel_map(score_chunk, jobs, func_cpu=1, func_ram=2, grow=True)
print(f"processed {sum(r['rows'] for r in chunk_results):,} reviews")

### What just happened?

You just streamed ~500 MB of JSONL reviews from HuggingFace directly into 10 cloud workers, scored every review for profanity / caps / rage-mismatch, and pulled the top-K per bucket back to the client. The worker results are tiny (≤30 reviews per bucket), so the reduce step on the client is instant.

The full 571 M-review version does the same thing across all 34 Amazon categories on 500+ workers — see `scale.py` and `pipeline.py` in this repo.

### Inspect the top-10 "unhinged" reviews globally

In [ ]:
def merge_topk(chunks, field, k=10):
    all_items = [(score, row) for c in chunks for (score, row) in c[field]]
    return sorted(all_items, key=lambda x: x[0], reverse=True)[:k]

print("=" * 80)
print("TOP 5 MOST PROFANE (Video Games)")
print("=" * 80)
for score, row in merge_topk(chunk_results, "top_profane", k=5):
    print(f"[{row['rating']}★ | {score} profanity hits]")
    print(f"  {row['text'][:220]}…")
    print()

print("=" * 80)
print("TOP 5 ALL-CAPS REVIEWS")
print("=" * 80)
for score, row in merge_topk(chunk_results, "top_screaming", k=5):
    print(f"[{row['rating']}★]")
    print(f"  {row['text'][:220]}…")
    print()

print("=" * 80)
print("TOP 5 RAGE-DISGUISED-AS-5-STARS")
print("=" * 80)
for score, row in merge_topk(chunk_results, "top_rage_5star", k=5):
    print(f"[{row['rating']}★ | rage score {score}]")
    print(f"  {row['text'][:220]}…")
    print()

### Try this next

- Crank `N_CHUNKS` to 100 and `demo_bytes` to `total_bytes` — the entire Video Games category (~3.5 GB, 4M reviews) on 100 workers.
- Swap the category URL: every `raw/review_categories/*.jsonl` on HF is available (Books, Home & Kitchen, etc.).
- Add an `exclamation_run_length` bucket to surface the `10,594 "!"s` style reviews the full run found.
- Remove `top_*` and return the full tagged reviews — the client-side reduce can handle ~500K rows per chunk at this scale.

## Thank you for trying Burla! ❤️

### Run the full 571M-review version

See [`scale.py`](https://github.com/Burla-Cloud/amazon-review-distiller/blob/main/scale.py) and [`pipeline.py`](https://github.com/Burla-Cloud/amazon-review-distiller/blob/main/pipeline.py) in this repo — 545 byte-range chunks × 34 categories × 500+ concurrent Burla workers.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev